# Melanoma & Nevus Dermoscopy Dataset - Imputation (Stratified Random Hot-Deck)

##### 1. Load dữ liệu train/val/test

In [1]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("data/splitted/train.csv")
df_val = pd.read_csv("data/splitted/val.csv")
df_test = pd.read_csv("data/splitted/test.csv")

print("Train:", df_train.shape)
print("Val  :", df_val.shape)
print("Test :", df_test.shape)

Train: (12696, 10)
Val  : (2752, 10)
Test : (2685, 10)


##### 2. Gộp nhóm attribution

Áp dụng cùng mapping đã dùng ở `02_EDA.ipynb` cho cả 3 tập (train/val/test), để dùng `attribution_grouped` làm điều kiện impute.

In [2]:
attribution_group_map = {
    'ViDIR Group, Department of Dermatology, Medical University of Vienna': 'Department of Dermatology, Medical University of Vienna',
    'Hospital Clínic de Barcelona': 'Hospital Clinic de Barcelona',
    'Department of Dermatology, Hospital Clínic de Barcelona': 'Hospital Clinic de Barcelona',
    'Sydney Melanoma Diagnostic Center at Royal Prince Alfred Hospital, Pascale Guitera': 'Royal Prince Alfred Hospital',
    'Sydney Melanoma Diagnostic Center at Royal Prince Alfred Hospital': 'Royal Prince Alfred Hospital',
    'Anonymous': 'Other Sources',
    'Dr. Julien Anriot': 'Other Sources',
}

for df_ in [df_train, df_val, df_test]:
    df_['attribution_grouped'] = df_['attribution'].replace(attribution_group_map)

df_train['attribution_grouped'].value_counts()

attribution_grouped
Hospital Clinic de Barcelona                                                                                    4098
Other Sources                                                                                                   3500
Department of Dermatology, Medical University of Vienna                                                         3000
Memorial Sloan Kettering Cancer Center                                                                          1435
Royal Prince Alfred Hospital                                                                                     358
Hospital Italiano de Buenos Aires                                                                                250
The University of Queensland Diamantina Institute, The University of Queensland, Dermatology Research Centre      55
Name: count, dtype: int64

##### 3. Xây dựng pool impute (chỉ từ `df_train`)

Pool = tập giá trị thật (không missing) dùng để random sample khi fill. Chỉ xây từ `df_train` để tránh leakage, sau đó dùng chung pool này cho cả train/val/test.

Chỉ điều kiện hóa theo `attribution_grouped` cho **cả 3 cột** (`age_approx`, `sex`, `anatom_site_general`), **không dùng `diagnosis`** — vì `diagnosis` chính là nhãn (target) của bài toán. Nếu dùng nhãn thật để quyết định cách impute feature cho val/test, đó là target leakage: lúc deploy thật, model chưa biết `diagnosis` của bệnh nhân mới nên không thể tra cứu theo nhãn để impute. Fallback khi pool theo `attribution_grouped` rỗng/quá thưa → pool toàn cục.

In [3]:
def build_pool_1level(df, col, col_a):
    """Pool theo col_a, dict {val_a: array giá trị thật}."""
    non_missing = df[df[col].notna()]
    return non_missing.groupby(col_a)[col].apply(lambda s: s.values).to_dict()


# --- age_approx ---
age_pool_attr = build_pool_1level(df_train, 'age_approx', 'attribution_grouped')
age_pool_global = df_train['age_approx'].dropna().values

# --- sex ---
sex_pool_attr = build_pool_1level(df_train, 'sex', 'attribution_grouped')
sex_pool_global = df_train['sex'].dropna().values

# --- anatom_site_general ---
anatom_pool_attr = build_pool_1level(df_train, 'anatom_site_general', 'attribution_grouped')
anatom_pool_global = df_train['anatom_site_general'].dropna().values

print("Số nhóm pool attribution_grouped cho age :", len(age_pool_attr))
print("Số nhóm pool attribution_grouped cho sex :", len(sex_pool_attr))
print("Số nhóm pool attribution_grouped cho anatom:", len(anatom_pool_attr))

Số nhóm pool attribution_grouped cho age : 7
Số nhóm pool attribution_grouped cho sex : 7
Số nhóm pool attribution_grouped cho anatom: 7


##### 4. Hàm impute (stratified random hot-deck, có fallback)

In [4]:
rng = np.random.default_rng(42)


def impute_1level(df, col, col_a, pool_full, pool_global, rng):
    """Fill NaN trong df[col] bằng random hot-deck, group theo col_a,
    fallback global."""
    result = df[col].copy()
    missing_idx = df.index[result.isna()]

    for idx in missing_idx:
        pool = pool_full.get(df.loc[idx, col_a])
        if pool is None or len(pool) == 0:
            pool = pool_global
        result.loc[idx] = rng.choice(pool)

    return result

##### 5. Áp dụng impute cho train/val/test

In [5]:
print("Missing trước khi impute:")
for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name}:", d[['age_approx', 'sex', 'anatom_site_general']].isna().sum().to_dict())

for d in [df_train, df_val, df_test]:
    d['age_approx'] = impute_1level(
        d, 'age_approx', 'attribution_grouped',
        age_pool_attr, age_pool_global, rng,
    )
    d['sex'] = impute_1level(
        d, 'sex', 'attribution_grouped',
        sex_pool_attr, sex_pool_global, rng,
    )
    d['anatom_site_general'] = impute_1level(
        d, 'anatom_site_general', 'attribution_grouped',
        anatom_pool_attr, anatom_pool_global, rng,
    )

print("\nMissing sau khi impute:")
for name, d in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name}:", d[['age_approx', 'sex', 'anatom_site_general']].isna().sum().to_dict())

Missing trước khi impute:
  train: {'age_approx': 324, 'sex': 274, 'anatom_site_general': 785}
  val: {'age_approx': 73, 'sex': 77, 'anatom_site_general': 191}
  test: {'age_approx': 61, 'sex': 46, 'anatom_site_general': 159}



Missing sau khi impute:
  train: {'age_approx': 0, 'sex': 0, 'anatom_site_general': 0}
  val: {'age_approx': 0, 'sex': 0, 'anatom_site_general': 0}
  test: {'age_approx': 0, 'sex': 0, 'anatom_site_general': 0}


In [6]:
df_train[['age_approx', 'sex', 'anatom_site_general', 'diagnosis']]

,age_approx,sex,anatom_site_general,diagnosis
0,60.0,female,upper extremity,melanoma
1,80.0,male,posterior torso,melanoma
2,30.0,female,posterior torso,melanoma
3,30.0,female,lower extremity,melanoma
4,55.0,female,lower extremity,melanoma
...,...,...,...,...
12691,55.0,male,posterior torso,melanoma
12692,40.0,male,upper extremity,nevus
12693,60.0,male,posterior torso,nevus
12694,50.0,female,upper extremity,nevus


##### 6. Feature engineering — ngưỡng tuổi U50/U60/U70

Tạo 3 cột binary từ `age_approx` (sau impute) để xem mốc tuổi nào tác động mạnh hơn đến `diagnosis`, dựa theo vùng chênh lệch rõ rệt đã thấy ở `bin_age_by_decade` (51-60 → 61-70 → 71-80).

In [7]:
for d in [df_train, df_val, df_test]:
    d['U50'] = (d['age_approx'] > 50).astype(int)
    d['U60'] = (d['age_approx'] > 60).astype(int)
    d['U70'] = (d['age_approx'] > 70).astype(int)

df_train[['age_approx', 'U50', 'U60', 'U70']].head()

,age_approx,U50,U60,U70
0,60.0,1,0,0
1,80.0,1,1,1
2,30.0,0,0,0
3,30.0,0,0,0
4,55.0,1,0,0


##### 7. Encoding

- `sex`: binary encoding (male=1, female=0)
- `anatom_site_general`: one-hot encoding (8 category, không có thứ tự tự nhiên nên không dùng ordinal/target encoding để tránh leakage từ nhãn)

In [8]:
from sklearn.preprocessing import OneHotEncoder

# --- sex: binary encoding ---
for d in [df_train, df_val, df_test]:
    d['sex_encoded'] = (d['sex'] == 'male').astype(int)

# --- anatom_site_general: one-hot encoding, fit categories CHỈ trên train ---
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(df_train[['anatom_site_general']])

anatom_cols = [f"anatom_{c}" for c in ohe.categories_[0]]

for d in [df_train, df_val, df_test]:
    encoded = ohe.transform(d[['anatom_site_general']])
    d[anatom_cols] = encoded.astype(int)

df_train[['sex', 'sex_encoded'] + ['anatom_site_general'] + anatom_cols].head()

,sex,sex_encoded,anatom_site_general,anatom_anterior torso,anatom_head/neck,anatom_lateral torso,anatom_lower extremity,anatom_oral/genital,anatom_palms/soles,anatom_posterior torso,anatom_upper extremity
0,female,0,upper extremity,0,0,0,0,0,0,0,1
1,male,1,posterior torso,0,0,0,0,0,0,1,0
2,female,0,posterior torso,0,0,0,0,0,0,1,0
3,female,0,lower extremity,0,0,0,1,0,0,0,0
4,female,0,lower extremity,0,0,0,1,0,0,0,0


##### 8. Scaling `age_approx`

Fit `StandardScaler` chỉ trên `df_train`, áp dụng (transform) xuống cả val/test — không fit riêng cho val/test, tránh leakage.

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(df_train[['age_approx']])

for d in [df_train, df_val, df_test]:
    d['age_approx_scaled'] = scaler.transform(d[['age_approx']])

df_train[['age_approx', 'age_approx_scaled']].describe()

,age_approx,age_approx_scaled
count,12696.000000,1.269600e+04
mean,52.019140,-4.477270e-17
std,17.761747,1.000039e+00
min,5.000000,-2.647318e+00
25%,40.000000,-6.767135e-01
50%,50.000000,-1.136836e-01
75%,65.000000,7.308612e-01
max,85.000000,1.856921e+00


##### 9. Lưu dữ liệu đã xử lý

In [10]:
df_train.to_csv("data/processed/train.csv", index=False)
df_val.to_csv("data/processed/val.csv", index=False)
df_test.to_csv("data/processed/test.csv", index=False)

print("Đã lưu: data/processed/train.csv, val.csv, test.csv")

Đã lưu: data/processed/train.csv, val.csv, test.csv
